# ViT-B/16 segmentation on BTXRD (20 epochs)

Train binary tumor segmentation with a ViT-B/16 backbone using a Kaggle GPU. The original BTXRD splits are filtered to remove conflicting labels and training images duplicated in validation or test. The dataset and pretrained weights are downloaded from Hugging Face. Training runs for 20 epochs and saves `best.pt` and `final.pt` in `/kaggle/working/btxrd_vit_20e/`. The test split is reserved for final evaluation. The project source code is bundled in this notebook.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

assert Path('/kaggle/working').is_dir(), 'This notebook must run on Kaggle.'
os.environ['HF_HOME'] = '/kaggle/temp/huggingface'
os.environ['PYTHONUNBUFFERED'] = '1'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm==1.0.31'], check=True)
import timm
import torch
from huggingface_hub import hf_hub_download

assert torch.cuda.is_available(), 'CUDA GPU unavailable: check the Kaggle accelerator and GPU quota.'
print('Torch:', torch.__version__, 'timm:', timm.__version__)
print('GPU:', torch.cuda.get_device_name(0), flush=True)


In [ ]:
import hashlib

DATASET_REVISION = 'cf8b0569b18e43e7cecab412a06e60aeaf258b32'
EXPECTED_SHA256 = '59ce94244fdf997f0c68e6a378690a4359e6c411f7fda7ac2f3994489637884b'
archive = Path(hf_hub_download(repo_id='torusama/xla-seminar-uia', repo_type='dataset',
                               filename='btxrd_final.zip', revision=DATASET_REVISION))
digest = hashlib.sha256(archive.read_bytes()).hexdigest()
if digest != EXPECTED_SHA256:
    raise RuntimeError(f'Dataset checksum mismatch: {digest}')
print('BTXRD downloaded; SHA-256:', digest, flush=True)


In [ ]:
# Project sources are bundled in this notebook; no Git checkout is needed.
PROJECT = Path('/kaggle/temp/xla-btxrd-vit')
SOURCES = {'src/xla/__init__.py': '"""Research scaffold for vision classification and segmentation."""\n\n__version__ = "0.1.0"\n', 'src/xla/data/__init__.py': '"""Data interfaces for future phases."""\n', 'src/xla/data/hf_segmentation.py': '"""Read the pinned Hugging Face archive and rasterize polygon masks."""\nimport hashlib\nimport io\nimport json\nimport os\nfrom pathlib import Path\nfrom zipfile import ZipFile\n\nimport torch\nfrom PIL import Image, ImageDraw\nfrom torch.utils.data import Dataset\nfrom torchvision.transforms import functional as TF\n\n\nclass PolygonArchive(Dataset):\n    """Binary union of all annotated polygons; normal samples have empty masks."""\n\n    def __init__(self, archive: Path, split: str, excluded_hashes: set[str] | None = None):\n        self.archive = Path(archive)\n        self.samples = []\n        self.excluded = 0\n        self._zip = None\n        self._zip_pid = None\n        excluded_hashes = excluded_hashes or set()\n        with ZipFile(self.archive) as z:\n            names = set(z.namelist())\n            annotations = sorted(n for n in names if n.startswith(f"{split}/annotations/")\n                                 and n.endswith(".json"))\n            for name in annotations:\n                annotation = json.loads(z.read(name))\n                category = Path(name).parent.name\n                image_name = f"{split}/images/{category}/{annotation[\'image_name\']}"\n                if image_name not in names:\n                    raise ValueError(f"Missing image for {name}: {image_name}")\n                digest = hashlib.sha256(z.read(image_name)).hexdigest()\n                if digest in excluded_hashes:\n                    self.excluded += 1\n                    continue\n                self.samples.append((image_name, name, digest))\n        if not self.samples:\n            raise ValueError(f"No samples in {self.archive} split {split}")\n\n    @property\n    def hashes(self) -> set[str]:\n        return {sample[2] for sample in self.samples}\n\n    def __len__(self):\n        return len(self.samples)\n\n    def __getstate__(self):\n        state = self.__dict__.copy()\n        state["_zip"] = None\n        state["_zip_pid"] = None\n        return state\n\n    def __getitem__(self, index):\n        image_name, annotation_name, _ = self.samples[index]\n        if self._zip is None or self._zip_pid != os.getpid():\n            if self._zip is not None:\n                self._zip.close()\n            self._zip = ZipFile(self.archive)\n            self._zip_pid = os.getpid()\n        annotation = json.loads(self._zip.read(annotation_name))\n        with Image.open(io.BytesIO(self._zip.read(image_name))) as source:\n            image = source.convert("RGB")\n        expected = (annotation["image_width"], annotation["image_height"])\n        if image.size != expected or expected != (224, 224):\n            raise ValueError(f"Image and annotation dimensions do not match 224x224: {image_name}")\n        mask = Image.new("L", image.size, 0)\n        draw = ImageDraw.Draw(mask)\n        count = 0\n        for item in annotation["annotations"]:\n            for polygon in item.get("polygons", []):\n                if len(polygon) < 3:\n                    raise ValueError(f"Invalid polygon: {annotation_name}")\n                draw.polygon([tuple(point) for point in polygon], fill=1)\n                count += 1\n        if not count and annotation["class_label"].lower() != "normal":\n            raise ValueError(f"Non-normal sample has no polygons: {annotation_name}")\n        pixels = TF.normalize(TF.to_tensor(image), mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))\n        target = torch.frombuffer(bytearray(mask.tobytes()), dtype=torch.uint8).reshape(1, 224, 224)\n        return pixels, target.float()\n\n\ndef make_splits(archive: Path):\n    """Remove conflicting labels and train duplicates of held-out images."""\n    splits = [PolygonArchive(archive, name) for name in ("train", "val", "test")]\n    categories = {}\n    for split in splits:\n        for image_name, _, digest in split.samples:\n            categories.setdefault(digest, set()).add(Path(image_name).parent.name)\n    conflicts = {digest for digest, labels in categories.items() if len(labels) > 1}\n    for split in splits:\n        original = len(split.samples)\n        split.samples = [sample for sample in split.samples if sample[2] not in conflicts]\n        split.excluded = original - len(split.samples)\n    train, val, test = splits\n    if val.hashes & test.hashes:\n        raise ValueError("Validation and test contain identical images; resolve the split first")\n    held_out = val.hashes | test.hashes\n    original = len(train.samples)\n    train.samples = [sample for sample in train.samples if sample[2] not in held_out]\n    train.excluded += original - len(train.samples)\n    if any(not split.samples for split in splits):\n        raise ValueError("A split is empty after removing conflicting and duplicate images")\n    return train, val, test\n', 'src/xla/models/__init__.py': '"""Models interfaces for future phases."""\n', 'src/xla/models/vit.py': '"""Thin adapter for the original ImageNet-21k ViT-B/16 PyTorch port."""\n\nMODEL_NAME = "vit_base_patch16_224.orig_in21k"\n\n\ndef build_vit(*, pretrained: bool = True, num_classes: int = 2):\n    """Create timm ViT; pretrained backbone may be downloaded by timm.\n\n    The checkpoint has no classification head, so timm initializes a new head.\n    pretrained=False is intended for offline tests, not a pretrained experiment.\n    """\n    if num_classes < 1:\n        raise ValueError("num_classes must be positive")\n    try:\n        import timm\n    except ImportError as exc:\n        raise RuntimeError(\'Install the ViT extra: python -m pip install -e ".[vit]"\') from exc\n    try:\n        return timm.create_model(MODEL_NAME, pretrained=pretrained, num_classes=num_classes)\n    except Exception as exc:\n        mode = "pretrained weights" if pretrained else "model"\n        raise RuntimeError(f"Could not load {mode} for {MODEL_NAME}: {exc}") from exc\n\n\ndef preprocessing_config(model) -> dict:\n    """Return the model\'s actual timm preprocessing settings."""\n    from timm.data import resolve_model_data_config\n\n    return resolve_model_data_config(model)\n', 'src/xla/models/vit_segmentation.py': '"""A simple dense prediction head on the original ViT-B/16 backbone."""\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nfrom xla.models.vit import build_vit\n\n\nclass ViTSegmentation(nn.Module):\n    """Predict one foreground logit per pixel from ViT patch tokens."""\n\n    def __init__(self, pretrained: bool = True):\n        super().__init__()\n        self.backbone = build_vit(pretrained=pretrained, num_classes=2)\n        self.backbone.reset_classifier(0)\n        self.head = nn.Sequential(\n            nn.Conv2d(self.backbone.embed_dim, 256, kernel_size=3, padding=1),\n            nn.GELU(),\n            nn.Conv2d(256, 1, kernel_size=1),\n        )\n\n    def forward(self, images: torch.Tensor) -> torch.Tensor:\n        tokens = self.backbone.forward_features(images)\n        tokens = tokens[:, self.backbone.num_prefix_tokens:, :]\n        height, width = self.backbone.patch_embed.grid_size\n        features = tokens.transpose(1, 2).reshape(images.shape[0], -1, height, width)\n        return F.interpolate(self.head(features), size=images.shape[-2:],\n                             mode="bilinear", align_corners=False)\n', 'src/xla/engine/__init__.py': '"""Engine interfaces for future phases."""\n', 'src/xla/engine/vit_segmentation.py': '"""Training and evaluation utilities for binary ViT segmentation."""\nimport torch\nfrom torch.nn import functional as F\n\n\ndef segmentation_loss(logits, target, positive_weight=10.0):\n    probabilities = logits.sigmoid()\n    if positive_weight is None:\n        intersection = (probabilities * target).sum(dim=(1, 2, 3))\n        denominator = probabilities.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))\n        return F.binary_cross_entropy_with_logits(logits, target) + (\n            1 - ((2 * intersection + 1) / (denominator + 1)).mean()\n        )\n    intersection = (probabilities * target).sum()\n    denominator = probabilities.sum() + target.sum()\n    dice_loss = 1 - (2 * intersection + 1) / (denominator + 1)\n    weight = logits.new_tensor(positive_weight)\n    return F.binary_cross_entropy_with_logits(logits, target, pos_weight=weight) + dice_loss\n\n\ndef run_epoch(model, loader, device, optimizer=None, positive_weight=10.0):\n    training = optimizer is not None\n    model.train(training)\n    total_loss = 0.0\n    tp = fp = fn = 0\n    count = 0\n    with torch.set_grad_enabled(training):\n        for images, target in loader:\n            images, target = images.to(device), target.to(device)\n            if training:\n                optimizer.zero_grad(set_to_none=True)\n            logits = model(images)\n            loss = segmentation_loss(logits, target, positive_weight)\n            if training:\n                loss.backward()\n                optimizer.step()\n            batch = images.shape[0]\n            total_loss += loss.item() * batch\n            count += batch\n            prediction = logits.detach() >= 0\n            truth = target.bool()\n            tp += (prediction & truth).sum().item()\n            fp += (prediction & ~truth).sum().item()\n            fn += (~prediction & truth).sum().item()\n    return {"loss": total_loss / count, "dice": (2 * tp / (2 * tp + fp + fn)) if tp + fp + fn else 1.0,\n            "iou": (tp / (tp + fp + fn)) if tp + fp + fn else 1.0}\n', 'scripts/train_vit_segmentation.py': '"""Train or evaluate a ViT segmentation model on the downloaded archives."""\nimport argparse\nimport hashlib\nimport json\nimport random\nfrom pathlib import Path\n\nimport torch\nfrom torch.utils.data import DataLoader\n\nfrom xla.data.hf_segmentation import make_splits\nfrom xla.engine.vit_segmentation import run_epoch\nfrom xla.models.vit_segmentation import ViTSegmentation\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with path.open("rb") as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    commands = parser.add_subparsers(dest="command", required=True)\n    for name in ("train", "evaluate"):\n        command = commands.add_parser(name)\n        command.add_argument("--archive", type=Path, required=True)\n        command.add_argument("--batch-size", type=int, default=4)\n        command.add_argument("--workers", type=int, default=0)\n        command.add_argument("--device", choices=("auto", "cpu", "cuda"), default="auto")\n    train = commands.choices["train"]\n    train.add_argument("--output", type=Path, required=True)\n    train.add_argument("--epochs", type=int, default=20)\n    train.add_argument("--learning-rate", type=float, default=0.0001)\n    train.add_argument("--positive-weight", type=float, default=10.0)\n    train.add_argument("--seed", type=int, default=42)\n    train.add_argument("--from-scratch", action="store_true")\n    commands.choices["evaluate"].add_argument("--checkpoint", type=Path, required=True)\n    args = parser.parse_args()\n    if args.batch_size < 1 or args.workers < 0:\n        parser.error("batch-size must be positive and workers must be nonnegative")\n    if args.command == "train" and (args.epochs < 1 or args.learning_rate <= 0\n                                    or args.positive_weight <= 0):\n        parser.error("epochs, learning-rate and positive-weight must be positive")\n    device = torch.device("cuda" if args.device == "auto" and torch.cuda.is_available()\n                          else "cpu" if args.device == "auto" else args.device)\n    if device.type == "cuda" and not torch.cuda.is_available():\n        parser.error("CUDA is not available")\n    digest = sha256(args.archive)\n    train_set, val_set, test_set = make_splits(args.archive)\n    print(f"Samples: train={len(train_set)}, val={len(val_set)}, test={len(test_set)}; "\n          f"excluded: train={train_set.excluded}, val={val_set.excluded}, "\n          f"test={test_set.excluded}", flush=True)\n    if args.command == "evaluate":\n        checkpoint = torch.load(args.checkpoint, map_location="cpu", weights_only=True)\n        if checkpoint["archive_sha256"] != digest:\n            raise ValueError("Checkpoint was trained on a different archive")\n        model = ViTSegmentation(pretrained=False).to(device)\n        model.load_state_dict(checkpoint["model"])\n        loader = DataLoader(test_set, batch_size=args.batch_size, num_workers=args.workers)\n        positive_weight = checkpoint.get("settings", {}).get("positive_weight")\n        print(json.dumps(run_epoch(model, loader, device, positive_weight=positive_weight), indent=2))\n        return\n    if args.output.exists():\n        raise FileExistsError(f"Output directory already exists: {args.output}")\n    random.seed(args.seed)\n    torch.manual_seed(args.seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(args.seed)\n    model = ViTSegmentation(pretrained=not args.from_scratch).to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=args.learning_rate, weight_decay=0.01)\n    train_loader = DataLoader(train_set, batch_size=args.batch_size, shuffle=True,\n                              num_workers=args.workers)\n    val_loader = DataLoader(val_set, batch_size=args.batch_size, num_workers=args.workers)\n    args.output.mkdir(parents=True, exist_ok=False)\n    history = []\n    best = -1.0\n    settings = {"archive": str(args.archive), "archive_sha256": digest, "seed": args.seed,\n                "epochs": args.epochs, "batch_size": args.batch_size,\n                "learning_rate": args.learning_rate, "positive_weight": args.positive_weight,\n                "pretrained": not args.from_scratch,\n                "excluded_train": train_set.excluded, "excluded_val": val_set.excluded,\n                "excluded_test": test_set.excluded}\n    (args.output / "config.json").write_text(json.dumps(settings, indent=2), encoding="utf-8")\n    for epoch in range(1, args.epochs + 1):\n        training = run_epoch(model, train_loader, device, optimizer, args.positive_weight)\n        validation = run_epoch(model, val_loader, device, positive_weight=args.positive_weight)\n        record = {"epoch": epoch, "train": training, "val": validation}\n        history.append(record)\n        print(json.dumps(record), flush=True)\n        (args.output / "history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")\n        checkpoint = {"model": model.state_dict(), "epoch": epoch, "val": validation,\n                      "archive_sha256": digest, "settings": settings}\n        if validation["dice"] > best:\n            best = validation["dice"]\n            torch.save(checkpoint, args.output / "best.pt")\n        if epoch == args.epochs:\n            torch.save(checkpoint, args.output / "final.pt")\n\n\nif __name__ == "__main__":\n    main()\n'}
for name, content in SOURCES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')
os.environ['PYTHONPATH'] = str(PROJECT / 'src')
print('Project sources prepared:', len(SOURCES), 'files', flush=True)


In [ ]:
# Exactly 20 epochs. best.pt is selected on validation; final.pt is epoch 20.
output = Path('/kaggle/working/btxrd_vit_20e')
command = [sys.executable, '-u', str(PROJECT / 'scripts/train_vit_segmentation.py'), 'train',
           '--archive', str(archive), '--output', str(output), '--epochs', '20',
           '--batch-size', '4', '--learning-rate', '0.0001', '--positive-weight', '10', '--workers', '2',
           '--seed', '42', '--device', 'cuda']
subprocess.run(command, check=True, env=os.environ.copy())
assert (output / 'best.pt').is_file() and (output / 'final.pt').is_file()
print('Finished 20 epochs. Output files:', [p.name for p in output.iterdir()], flush=True)


After the notebook finishes, download `best.pt`, `final.pt`, `history.json`, and `config.json` from the notebook Output. `best.pt` is selected using validation Dice; `final.pt` contains the weights after epoch 20. Evaluate the test split only after choosing the model.
